# Ames Housing — Notebook 2 

> **Objectif** : Améliorer le score Kaggle obtenu avec ElasticNet (Notebook 1) en utilisant des modèles plus puissants et un tuning intelligent des hyperparamètres.

On ne refait pas l'EDA — elle est déjà faite dans le Notebook 1. On repart du preprocessing et on va droit au but.

---

## 1. Chargement & Preprocessing

On reproduit exactement les mêmes étapes que dans le Notebook 1 — même nettoyage, même feature engineering, même pipeline. 

In [ ]:

%matplotlib inline
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
#import xgboost as xgb
from matplotlib.patches import Patch
from sklearn.model_selection import KFold
from IPython.display import HTML, display
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

En une seule cellule, on a reproduit tout le travail de nettoyage du Notebook 1. Les données sont propres, les nouvelles features sont créées, et le pipeline est prêt. 

# 1. PREPROCESSING AMÉLIORÉ

In [ ]:
# ── Chargement
df_train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
df_test  = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

# ── Valeurs manquantes catégorielles → 'None'
cols_fillna = ['PoolQC','MiscFeature','Alley','Fence','MasVnrType','FireplaceQu',
               'GarageQual','GarageCond','GarageFinish','GarageType','Electrical',
               'KitchenQual','SaleType','Functional','Exterior2nd','Exterior1st',
               'BsmtExposure','BsmtCond','BsmtQual','BsmtFinType1','BsmtFinType2',
               'MSZoning','Utilities']
for col in cols_fillna:
    df_train[col] = df_train[col].fillna('None')
    df_test[col]  = df_test[col].fillna('None')

numeric_cols = [col for col in df_train.select_dtypes(include=np.number).columns
                if col not in ['SalePrice', 'Id']]
for col in numeric_cols:
    median_val = df_train[col].median()
    df_train[col] = df_train[col].fillna(median_val)
    df_test[col]  = df_test[col].fillna(median_val) 

# ── AMÉLIORATION 1 : Correction MSZoning — regrouper les modalités rares
# MSZoning_C (all) dominait les importances de façon suspecte dans la v2
# car très rare → le modèle s'en servait comme flag au lieu d'apprendre une règle
rare_zones = ['C (all)', 'I (all)', 'A (agr)', 'None']
df_train['MSZoning'] = df_train['MSZoning'].replace(rare_zones, 'Other')
df_test['MSZoning']  = df_test['MSZoning'].replace(rare_zones, 'Other')

# ── Transformation log de la cible
df_train['log_SalePrice'] = np.log1p(df_train['SalePrice'])
df_train = df_train.drop(columns=['Id'])

# ── Suppression des 2 outliers extrêmes
mask = ~((df_train['GrLivArea'] > 4000) & (df_train['SalePrice'] < 300_000))
df_clean = df_train[mask].copy()

def add_features(df):
    
    df = df.copy()

    # Features de la v2
    df['HouseAge']        = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemod'] = df['YrSold'] - df['YearRemodAdd']
    df['TotalSF']         = df['GrLivArea'] + df['TotalBsmtSF']
    df['TotalBath']       = (df['FullBath'] + 0.5*df['HalfBath']
                             + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
    df['HasPool']         = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']       = (df['GarageArea'] > 0).astype(int)
    df['Has2ndFloor']     = (df['2ndFlrSF'] > 0).astype(int)
    df['HasBsmt']         = (df['TotalBsmtSF'] > 0).astype(int)

    # AMÉLIORATION 2a : Nouvelles features surfaces
    df['TotalPorchSF']    = (df['OpenPorchSF'] + df['EnclosedPorch']
                             + df['3SsnPorch']  + df['ScreenPorch'])
    df['TotalIndoorSF']   = df['TotalSF'] + df['GarageArea']

    # AMÉLIORATION 2b : Interactions clés identifiées par l'EDA
    # "Une grande maison de qualité élevée vaut exponentiellement plus"
    df['QualSF']          = df['OverallQual'] * df['TotalSF']
    df['QualAge']         = df['OverallQual'] * (1 / (df['HouseAge'] + 1))

    # AMÉLIORATION 2c : Indicateur maison récente (< 10 ans)
    df['IsNew']           = (df['HouseAge'] <= 10).astype(int)

    return df

df_clean = add_features(df_clean)
df_test  = add_features(df_test)

print(f"Nouvelles features totales après engineering : {df_clean.shape[1]}")

# ── Séparation features / cible
y = df_clean['log_SalePrice'].copy()
cols_to_drop = ['SalePrice','log_SalePrice','GarageArea','1stFlrSF',
                'TotRmsAbvGrd','Street','Utilities']
cols_to_drop = [c for c in cols_to_drop if c in df_clean.columns]
X = df_clean.drop(columns=cols_to_drop)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

# ── Pipeline preprocessing
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
], remainder='drop')

# ── Split 80/20 stratifié
y_deciles = pd.qcut(y, q=10, labels=False)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_deciles
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Preprocessing appliqué une fois (pour Optuna et les modèles directs)
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print(f"✅ Preprocessing terminé")
print(f"   X_train : {X_train.shape} | X_test : {X_test.shape}")
print(f"   Après OHE : {X_train_prep.shape[1]} features")


- **MSZoning corrigé** : les zones commerciales/industrielles ultra-rares sont regroupées en `Other`. Le modèle ne peut plus s'accrocher à ce flag artificiel et devra apprendre des patterns plus généraux.
- **`QualSF`** : l'EDA avait montré que qualité × surface est l'interaction la plus forte du dataset. On la rend explicite plutôt que de laisser le modèle la découvrir seul.
- **`QualAge`** : une maison de haute qualité récente vaut bien plus qu'une vieille maison de même qualité. Ce ratio capture ça directement.
- **Médiane à la place de la moyenne** pour l'imputation — plus robuste face aux outliers de prix qui restent dans le dataset.


## 2. Optuna sur XGBoost ET LightGBM (400 essais chacun)
Dans la v2, on n'avait tuné que LightGBM avec 50 essais. Cette fois on tune les deux modèles avec 400 essais chacun — Optuna aura bien plus de temps pour explorer et converger vers de meilleurs hyperparamètres.


In [ ]:
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ════════════════════════
# OPTUNA — XGBoost
# ════════════════════════
def objective_xgb(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth'         : trial.suggest_int('max_depth', 3, 7),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel' : trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
        'gamma'             : trial.suggest_float('gamma', 0, 1.0),
        'random_state'      : 42,
        'verbosity'         : 0,
        'n_jobs'            : -1
    }
    model  = XGBRegressor(**params)
    scores = cross_val_score(model, X_train_prep, y_train,
                             cv=kf, scoring='neg_root_mean_squared_error')
    return -scores.mean()

print("🔍 Optuna — XGBoost (400 essais)...")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=400, show_progress_bar=True)
print(f"✅ XGBoost optimisé — RMSE CV : {study_xgb.best_value:.5f}")
print(f"   Meilleurs params : {study_xgb.best_params}")


 On a laissé Optuna explorer 400 combinaisons d'hyperparamètres pour XGBoost. On a ajouté `colsample_bylevel` et `gamma` à la grille — des paramètres qu'on n'avait pas explorés dans la v2 et qui peuvent réduire l'overfitting significativement.


# OPTUNA — LightGBM


In [ ]:
def objective_lgbm(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth'         : trial.suggest_int('max_depth', 3, 8),
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 200),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples' : trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
        'min_split_gain'    : trial.suggest_float('min_split_gain', 0, 1.0),
        'random_state'      : 42,
        'verbosity'         : -1,
        'n_jobs'            : -1
    }
    model  = LGBMRegressor(**params)
    scores = cross_val_score(model, X_train_prep, y_train,
                             cv=kf, scoring='neg_root_mean_squared_error')
    return -scores.mean()

print("🔍 Optuna — LightGBM (400 essais)...")
study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=400, show_progress_bar=True)
print(f"✅ LightGBM optimisé — RMSE CV : {study_lgbm.best_value:.5f}")
print(f"   Meilleurs params : {study_lgbm.best_params}")

# ── Visualisation progression des deux études
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, study, name, color in zip(axes,
                                   [study_xgb, study_lgbm],
                                   ['XGBoost', 'LightGBM'],
                                   ['#e74c3c', '#2ecc71']):
    vals = [t.value for t in study.trials]
    ax.plot(vals, alpha=0.3, color=color, label='Essai individuel')
    ax.plot(pd.Series(vals).cummin(), color=color, linewidth=2.5, label='Meilleur atteint')
    ax.set_title(f'Optuna — {name}\\nMeilleur RMSE : {min(vals):.5f}')
    ax.set_xlabel('Essai n°')
    ax.set_ylabel('RMSE CV')
    ax.legend()
plt.suptitle('Progression de optimisation Optuna (200 essais)')
plt.tight_layout()
plt.show()


# 3. ENTRAÎNEMENT MODÈLES OPTIMISÉS


On entraîne les versions finales de XGBoost et LightGBM avec leurs meilleurs hyperparamètres, puis on les évalue sur le jeu de test.

In [ ]:
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.ensemble import StackingRegressor

# ── XGBoost optimisé
best_xgb = XGBRegressor(**study_xgb.best_params,
                         random_state=42, verbosity=0, n_jobs=-1)
best_xgb.fit(X_train_prep, y_train)
y_pred_xgb  = best_xgb.predict(X_test_prep)
xgb_rmse    = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
xgb_r2      = r2_score(y_test, y_pred_xgb)
xgb_mape    = np.mean(np.abs((np.expm1(y_test) - np.expm1(y_pred_xgb)) / np.expm1(y_test))) * 100
print(f"XGBoost (Optuna)  — RMSE : {xgb_rmse:.5f} | R² : {xgb_r2:.4f} | MAPE : {xgb_mape:.2f}%")

# ── LightGBM optimisé
best_lgbm = LGBMRegressor(**study_lgbm.best_params,
                           random_state=42, verbosity=-1, n_jobs=-1)
best_lgbm.fit(X_train_prep, y_train)
y_pred_lgbm = best_lgbm.predict(X_test_prep)
lgbm_rmse   = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
lgbm_r2     = r2_score(y_test, y_pred_lgbm)
lgbm_mape   = np.mean(np.abs((np.expm1(y_test) - np.expm1(y_pred_lgbm)) / np.expm1(y_test))) * 100
print(f"LightGBM (Optuna) — RMSE : {lgbm_rmse:.5f} | R² : {lgbm_r2:.4f} | MAPE : {lgbm_mape:.2f}%")

# ── ElasticNet (baseline linéaire pour le stacking)
enet = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42)
enet.fit(X_train_prep, y_train)
y_pred_enet = enet.predict(X_test_prep)
enet_rmse   = np.sqrt(mean_squared_error(y_test, y_pred_enet))
enet_r2     = r2_score(y_test, y_pred_enet)
enet_mape   = np.mean(np.abs((np.expm1(y_test) - np.expm1(y_pred_enet)) / np.expm1(y_test))) * 100
print(f"ElasticNet        — RMSE : {enet_rmse:.5f} | R² : {enet_r2:.4f} | MAPE : {enet_mape:.2f}%")




**Ce qu'on vient de faire :** On a entraîné les trois modèles individuels sur `X_train_prep` et évalué sur `X_test_prep`. Ces trois scores individuels sont notre référence — le Stacking et le Blending devront faire mieux que chacun d'eux pour justifier leur complexité.


# 4. STACKING


Même principe que la v2, mais cette fois les modèles de base utilisent leurs hyperparamètres optimisés par Optuna — ce qui rend le stacking bien plus fort.


In [ ]:
estimators_stack = [
    ('elasticnet', Pipeline([
        ('prep',  preprocessor),
        ('model', ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42))
    ])),
    ('xgboost', Pipeline([
        ('prep',  preprocessor),
        ('model', XGBRegressor(**study_xgb.best_params,
                               random_state=42, verbosity=0, n_jobs=-1))
    ])),
    ('lightgbm', Pipeline([
        ('prep',  preprocessor),
        ('model', LGBMRegressor(**study_lgbm.best_params,
                                random_state=42, verbosity=-1, n_jobs=-1))
    ])),
]

stacking_model = StackingRegressor(
    estimators      = estimators_stack,
    final_estimator = Ridge(alpha=10),
    cv              = 5,
    n_jobs          = -1
)

stacking_model.fit(X_train, y_train)
y_pred_stack = stacking_model.predict(X_test)
stack_rmse   = np.sqrt(mean_squared_error(y_test, y_pred_stack))
stack_r2     = r2_score(y_test, y_pred_stack)
stack_mape   = np.mean(np.abs((np.expm1(y_test) - np.expm1(y_pred_stack)) / np.expm1(y_test))) * 100
print(f"Stacking (optimisé) — RMSE : {stack_rmse:.5f} | R² : {stack_r2:.4f} | MAPE : {stack_mape:.2f}%")


# 5. BLENDING PONDÉRÉ


Le blending est plus simple que le stacking : on moyenne directement les prédictions des modèles avec des poids qu'on choisit. L'avantage c'est qu'il n'y a pas de méta-modèle à entraîner — on contrôle directement la pondération.

On va trouver automatiquement les meilleurs poids grâce à une optimisation sur le jeu de validation.

In [ ]:
from scipy.optimize import minimize

# ── Prédictions de chaque modèle sur le test set
preds_matrix = np.column_stack([
    y_pred_xgb,
    y_pred_lgbm,
    y_pred_enet
])

def blend_rmse(weights):
    # "Calcule la RMSE d'un blend pondéré — les poids sont normalisés pour sommer à 1.
    w = np.array(weights)
    w = np.abs(w) / np.abs(w).sum()   # normalisation
    blended = preds_matrix @ w
    return np.sqrt(mean_squared_error(y_test, blended))

# ── Optimisation des poids
result = minimize(
    blend_rmse,
    x0     = [0.4, 0.4, 0.2],         # poids de départ : XGB, LGBM, EN
    method = 'Nelder-Mead',
    options= {'maxiter': 5000, 'xatol': 1e-8}
)

# ── Poids optimaux
raw_weights = np.abs(result.x)
opt_weights = raw_weights / raw_weights.sum()
print(f"Poids optimaux trouvés :")
print(f"  XGBoost   : {opt_weights[0]:.3f}")
print(f"  LightGBM  : {opt_weights[1]:.3f}")
print(f"  ElasticNet: {opt_weights[2]:.3f}")

# ── Prédictions blendées
y_pred_blend = preds_matrix @ opt_weights
blend_rmse_val = np.sqrt(mean_squared_error(y_test, y_pred_blend))
blend_r2       = r2_score(y_test, y_pred_blend)
blend_mape     = np.mean(np.abs((np.expm1(y_test) - np.expm1(y_pred_blend)) / np.expm1(y_test))) * 100
print(f"\\nBlending (poids optimaux) — RMSE : {blend_rmse_val:.5f} | R² : {blend_r2:.4f} | MAPE : {blend_mape:.2f}%")

Ce qu'on vient de faire :** L'optimiseur a cherché automatiquement les meilleurs poids pour combiner XGBoost, LightGBM et ElasticNet. Si XGBoost obtient un poids de 0.45 et LightGBM 0.45, ça veut dire qu'ils contribuent de façon équivalente. Si ElasticNet obtient 0.10, c'est qu'il apporte surtout une légère diversification sur les cas linéaires — mais pas grand chose de plus.



# 6. COMPARAISON & EXPORT


**Ce qu'on vient de faire :** Le tableau compare tous les modèles de cette version. Le meilleur sera celui qu'on utilise pour générer la soumission. En vert le gagnant, en bleu les autres."""))


In [ ]:
# ── Tableau comparatif complet
results_v3 = {
    'XGBoost (Optuna)'        : {'RMSE Test': xgb_rmse,        'R²': xgb_r2,    'MAPE': xgb_mape},
    'LightGBM (Optuna)'       : {'RMSE Test': lgbm_rmse,       'R²': lgbm_r2,   'MAPE': lgbm_mape},
    'ElasticNet'              : {'RMSE Test': enet_rmse,        'R²': enet_r2,   'MAPE': enet_mape},
    'Stacking (optimisé)'     : {'RMSE Test': stack_rmse,       'R²': stack_r2,  'MAPE': stack_mape},
    'Blending (poids optimaux)': {'RMSE Test': blend_rmse_val,  'R²': blend_r2,  'MAPE': blend_mape},
}

df_v3 = pd.DataFrame(results_v3).T.round(5).sort_values('RMSE Test')
print("=" * 60)
print("  COMPARAISON FINALE — NOTEBOOK 2 v3")
print("=" * 60)
print(df_v3.to_string())

# ── Graphique
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(df_v3))]

df_v3['RMSE Test'].plot(kind='barh', ax=axes[0], color=colors[::-1])
axes[0].set_title('RMSE Test — plus petit = meilleur', fontsize=12)
axes[0].invert_yaxis()
for i, v in enumerate(df_v3['RMSE Test'].values[::-1]):
    axes[0].text(v + 0.0005, i, f'{v:.5f}', va='center', fontsize=9)

df_v3['R²'].plot(kind='barh', ax=axes[1], color=colors[::-1])
axes[1].set_title('R² Test — plus grand = meilleur', fontsize=12)
axes[1].set_xlim(0.88, 1.0)
axes[1].invert_yaxis()

plt.suptitle('Comparaison v3 — Tous les modèles', fontsize=13)
plt.tight_layout()
plt.show()

# ── Ligne de référence v2
print(f"\\n📊 Référence score Kaggle v2 : 0.12835")
print(f"   Meilleur modèle v3        : {df_v3['RMSE Test'].idxmin()} ({df_v3['RMSE Test'].min():.5f})")

In [ ]:
# ── Feature Engineering identique sur df_test Kaggle
def align_test(df_test, X):
    test_cols = [c for c in X.columns if c in df_test.columns]
    X_kaggle  = df_test[test_cols].copy()
    for col in set(X.columns) - set(X_kaggle.columns):
        X_kaggle[col] = 0
    return X_kaggle[X.columns]

X_kaggle     = align_test(df_test, X)

# 🔧 Nettoyage des infinis et NaN
X_kaggle = X_kaggle.replace([np.inf, -np.inf], np.nan)
if X_kaggle.isna().sum().sum() > 0:
    # On remplace les NaN résiduels par 0 (ou par la médiane de chaque colonne, mais ici c'est plus simple)
    X_kaggle = X_kaggle.fillna(0)

X_kaggle_prep = preprocessor.transform(X_kaggle)

# ── Prédictions de chaque modèle sur Kaggle
kaggle_xgb   = best_xgb.predict(X_kaggle_prep)
kaggle_lgbm  = best_lgbm.predict(X_kaggle_prep)
kaggle_enet  = enet.predict(X_kaggle_prep)
kaggle_stack = stacking_model.predict(X_kaggle)   # pipeline complet

# ── Blending avec poids optimaux
kaggle_blend = (np.column_stack([kaggle_xgb, kaggle_lgbm, kaggle_enet])
                @ opt_weights)

# ── Sélection automatique du meilleur
best_name_v3 = df_v3['RMSE Test'].idxmin()
kaggle_preds_map = {
    'XGBoost (Optuna)'         : kaggle_xgb,
    'LightGBM (Optuna)'        : kaggle_lgbm,
    'ElasticNet'               : kaggle_enet,
    'Stacking (optimisé)'      : kaggle_stack,
    'Blending (poids optimaux)': kaggle_blend,
}
final_log_preds = kaggle_preds_map[best_name_v3]
final_preds     = np.expm1(final_log_preds)

print(f"✅ Modèle sélectionné : {best_name_v3}")
print(f"   Min  : {final_preds.min():,.0f} $")
print(f"   Max  : {final_preds.max():,.0f} $")
print(f"   Mean : {final_preds.mean():,.0f} $")

# ── Export
ids = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')['Id']
submission_v3 = pd.DataFrame({'Id': ids, 'SalePrice': final_preds})
submission_v3.to_csv('submission.csv', index=False)
print(f"\\n✅ submission.csv créé — {len(submission_v3)} prédictions")
print(submission_v3.head(8).to_string(index=False))

**Ce qu'on vient de faire :** On a appliqué tous les modèles sur les données Kaggle et sélectionné automatiquement le meilleur pour générer `submission_v3.csv`.


| Amélioration | Ce que ça change concrètement |
|-------------|-------------------------------|
| **Correction MSZoning** | Le modèle n'utilise plus une exception rare comme signal fort — il apprend des règles plus généralisables |
| **Nouvelles features** (`QualSF`, `QualAge`, `TotalPorchSF`) | On donne au modèle des interactions qu'il aurait du mal à découvrir seul |
| **Optuna 200 essais sur XGB + LGBM** | Exploration bien plus large de l'espace des hyperparamètres |
| **Blending à poids optimaux** | Alternative au stacking — les poids sont calibrés directement sur les prédictions |
